# 02. Baseline DummyClassifier

Este notebook entrena un baseline reproducible con `DummyClassifier`. No es un modelo de producción: permite comparar los modelos posteriores con una referencia que no aprende relaciones entre las features y el target.

## Configuración requerida

Antes de ejecutar, configure MLflow como se documenta en el README mediante `MLFLOW_TRACKING_URI` y, para servidores remotos, las credenciales y el Workspace aprobados. No escriba URLs ni credenciales en este notebook.

También entregue el contexto académico vigente desde InvoiceOps mediante estas variables no secretas: `INVOICEOPS_ORGANIZATION_SLUG`, `INVOICEOPS_OWNER_TYPE`, `INVOICEOPS_OWNER_ID` y `INVOICEOPS_CREATED_BY_RUT`. `INVOICEOPS_OWNER_ID` debe ser el UUID estable de `User.id` o `Group.id`; no derive identidades desde nombres ni slugs.

In [ ]:
import csv
import os

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from invoiceops_ml.data import MODEL_FEATURES, TARGET, generate_synthetic_dataset

SEED = 202605
ROWS = 12_000
dataset_dir = generate_synthetic_dataset(seed=SEED, rows=ROWS)


def load_split(name: str) -> tuple[list[list[str]], list[int]]:
    with (dataset_dir / f'{name}.csv').open(newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    features = [[row[feature] for feature in MODEL_FEATURES] for row in rows]
    target = [int(row[TARGET] == 'True') for row in rows]
    return features, target


train_features, train_target = load_split('train')
validation_features, validation_target = load_split('validation')
test_features, test_target = load_split('test')
len(train_target), len(validation_target), len(test_target)

## Entrenamiento y evaluación

`strategy='prior'` aprende solamente la prevalencia del target del train split. El baseline no aplica preprocessing ni registra un model artifact: esas responsabilidades se introducen con los modelos entrenables posteriores. La validación se usa para comparación; el test se registra como evaluación final separada.

In [ ]:
model = DummyClassifier(strategy="prior")
model.fit(train_features, train_target)


def classification_metrics(prefix: str, features: list[list[str]], target: list[int]) -> dict[str, float]:
    predictions = model.predict(features)
    return {
        f'{prefix}_accuracy': accuracy_score(target, predictions),
        f'{prefix}_precision': precision_score(target, predictions, zero_division=0),
        f'{prefix}_recall': recall_score(target, predictions, zero_division=0),
        f'{prefix}_f1': f1_score(target, predictions, zero_division=0),
    }


metrics = {
    **classification_metrics('validation', validation_features, validation_target),
    **classification_metrics('test', test_features, test_target),
}
metrics

## Registro en MLflow

El mismo notebook funciona contra MLflow local o remoto porque usa la configuración reusable de ML-01. Cada run recibe los tags de ownership de ML-02 para que la UI permita encontrarlo por organización y propietario.

In [ ]:
import mlflow

from invoiceops_ml.mlflow import configure_mlflow, mlflow_config_from_env
from invoiceops_ml.ownership import OwnershipContext, set_run_ownership_tags


def required_environment(name: str) -> str:
    value = os.environ.get(name, '').strip()
    if not value:
        raise ValueError(f'{name} must be set from the current InvoiceOps context')
    return value


configure_mlflow(mlflow_config_from_env())
ownership_context = OwnershipContext(
    organization_slug=required_environment('INVOICEOPS_ORGANIZATION_SLUG'),
    owner_type=required_environment('INVOICEOPS_OWNER_TYPE'),
    owner_id=required_environment('INVOICEOPS_OWNER_ID'),
    created_by_rut=required_environment('INVOICEOPS_CREATED_BY_RUT'),
)

with mlflow.start_run(run_name="dummy-baseline") as run:
    set_run_ownership_tags(ownership_context)
    mlflow.log_params(model.get_params())
    mlflow.log_metrics(metrics)

run.info.run_id

## Revisión

Abra el run en la UI de MLflow y confirme los parámetros, las métricas `validation_*` y `test_*`, y los tags `organization_slug`, `owner_type`, `owner_id` y `created_by_rut`. El siguiente notebook reemplazará este baseline por Logistic Regression con preprocessing dentro de un pipeline.